In [2]:
from pathlib import Path
import os
import pandas as pd
import subprocess
import shutil

HOME = Path("")
WS = HOME / "workspace"

print("Current directory:", os.getcwd())

print("\nBED files:")
for x in HOME.glob("*.bed"):
    print(" ", x)

print("\nMounted workspace resources:")
for i, x in enumerate(WS.iterdir()):
    print(i, x)

Current directory: /home/dataproc

BED files:
  hglft_genome_1a6dc1_c8f380.bed
  BMI_hg19.bed

Mounted workspace resources:
0 workspace/dataproc-staging-agemr
1 workspace/dataproc-temp-agemr
2 workspace/agemr-data


In [3]:
print("Checking writable workspace resources:")

writable = []

for x in WS.iterdir():
    if not x.is_dir():
        continue

    test_file = x / ".write_test"

    try:
        test_file.write_text("test")
        test_file.unlink()
        writable.append(x)
        print("WRITABLE :", x)
    except Exception:
        print("READ ONLY:", x)

Checking writable workspace resources:
WRITABLE : workspace/dataproc-staging-agemr
WRITABLE : workspace/dataproc-temp-agemr
WRITABLE : workspace/agemr-data


In [4]:
OUTROOT = Path("workspace/agemr-data")

In [5]:
lifted_files = list(HOME.glob("hglft_genome_*.bed"))

print(lifted_files)

assert len(lifted_files) == 1, \
    f"Expected 1 lifted BED file, found {len(lifted_files)}"

LIFTED = lifted_files[0]

bed = pd.read_csv(
    LIFTED,
    sep="\t",
    header=None,
    comment="#"
)

print("Raw shape:", bed.shape)
display(bed.head())

[PosixPath('hglft_genome_1a6dc1_c8f380.bed')]
Raw shape: (3024, 5)


,0,1,2,3,4
0,chr4,140174331,140174332,rs182612953,1
1,chr8,78580220,78580221,rs1983384,1
2,chr10,113331711,113331712,rs60640904,1
3,chr16,53768581,53768582,rs11642015,1
4,chr18,60181417,60181418,rs66922415,1


In [6]:
bed = bed.iloc[:, :4].copy()
bed.columns = ["chrom", "start0", "end0", "rsid"]

bed["chrom"] = bed["chrom"].astype(str)
bed["start0"] = bed["start0"].astype(int)
bed["end0"] = bed["end0"].astype(int)
bed["rsid"] = bed["rsid"].astype(str)

print("Number lifted:", len(bed))
print(bed["chrom"].value_counts().sort_index())

Number lifted: 3024
chrom
chr1                   261
chr10                  141
chr11                  130
chr12                  142
chr13                   90
chr14                   75
chr15                   68
chr16                  104
chr17                   86
chr18                   86
chr19                   64
chr2                   254
chr20                   87
chr21                   49
chr22                   36
chr3                   229
chr4                   228
chr5                   203
chr6                   209
chr7                   157
chr8                   205
chr8_KI270821v1_alt      1
chr9                   119
Name: count, dtype: int64


In [7]:
autosomes = {f"chr{i}" for i in range(1, 23)}

bed_main = bed.loc[bed["chrom"].isin(autosomes)].copy()

print("Primary autosomal SNPs:", len(bed_main))
print("Removed:", len(bed) - len(bed_main))

Primary autosomal SNPs: 3023
Removed: 1


In [8]:
EXTRACT_BED = HOME / "BMI_hg38_extract.bed"

bed_main[
    ["chrom", "start0", "end0", "rsid"]
].to_csv(
    EXTRACT_BED,
    sep="\t",
    header=False,
    index=False
)

In [9]:
manifest = bed_main.copy()
manifest["pos38"] = manifest["end0"]

MANIFEST = HOME / "BMI_hg38_manifest.tsv"

manifest[
    ["rsid", "chrom", "pos38", "start0", "end0"]
].to_csv(
    MANIFEST,
    sep="\t",
    index=False
)

display(manifest.head())
print("N SNP:", len(manifest))

,chrom,start0,end0,rsid,pos38
0,chr4,140174331,140174332,rs182612953,140174332
1,chr8,78580220,78580221,rs1983384,78580221
2,chr10,113331711,113331712,rs60640904,113331712
3,chr16,53768581,53768582,rs11642015,53768582
4,chr18,60181417,60181418,rs66922415,60181418


N SNP: 3023


In [13]:
for i, x in enumerate(WS.iterdir()):
    print(i, x)

0 workspace/dataproc-staging-agemr
1 workspace/dataproc-temp-agemr
2 workspace/agemr-data
3 workspace/Data from All of Us Controlled Tier 


In [14]:
!wb resource list-tree

├── Data from All of Us Controlled Tier 
│   ├── wgs_bmi_bp
│   ├── C2025Q4R6
│   └── vwb-aou-datasets-controlled-v9
├── JupyterLab_Spark_cluster_for_AoU_Spark_20260930
├── agemr-data
├── dataproc-staging-agemr
└── dataproc-temp-agemr


In [20]:
from pathlib import Path

GENOMICS = Path(
    "workspace/Data from All of Us Controlled Tier /"
    "vwb-aou-datasets-controlled-v9"
)

OUTROOT = Path("workspace/agemr-data")

print("GENOMICS exists:", GENOMICS.exists())
print("OUTROOT exists:", OUTROOT.exists())

GENOMICS exists: True
OUTROOT exists: True


In [17]:
!find "$HOME/workspace" -maxdepth 3 -mindepth 1 -type d -print

/home/dataproc/workspace/dataproc-staging-agemr
/home/dataproc/workspace/dataproc-staging-agemr/google-cloud-dataproc-metainfo
/home/dataproc/workspace/dataproc-staging-agemr/google-cloud-dataproc-metainfo/23d71dbb-7bad-4748-853f-410b30bb8023
/home/dataproc/workspace/dataproc-staging-agemr/google-cloud-dataproc-metainfo/9fa6b386-bde2-4d6e-90a1-201b5ae0ecb4
/home/dataproc/workspace/dataproc-staging-agemr/notebooks
/home/dataproc/workspace/dataproc-staging-agemr/notebooks/jupyter
/home/dataproc/workspace/dataproc-temp-agemr
/home/dataproc/workspace/dataproc-temp-agemr/23d71dbb-7bad-4748-853f-410b30bb8023
/home/dataproc/workspace/dataproc-temp-agemr/23d71dbb-7bad-4748-853f-410b30bb8023/mapreduce-job-history
/home/dataproc/workspace/dataproc-temp-agemr/23d71dbb-7bad-4748-853f-410b30bb8023/spark-job-history
/home/dataproc/workspace/dataproc-temp-agemr/9fa6b386-bde2-4d6e-90a1-201b5ae0ecb4
/home/dataproc/workspace/dataproc-temp-agemr/9fa6b386-bde2-4d6e-90a1-201b5ae0ecb4/mapreduce-job-history


In [21]:
for x in sorted(GENOMICS.iterdir()):
    print(x.name)

pooled
v7
v8
v9


In [22]:
V9 = GENOMICS / "v9"

print("V9 exists:", V9.exists())

for x in sorted(V9.iterdir()):
    print(x.name)

V9 exists: True
known_issues
microarray
multiomics
wgs


In [23]:
import subprocess
import shlex

v9q = shlex.quote(str(V9))

cmd = f"""
find {v9q} -maxdepth 3 -type d 2>/dev/null | sort
"""

res = subprocess.run(
    ["bash", "-lc", cmd],
    capture_output=True,
    text=True
)

print(res.stdout)

workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9
workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/known_issues
workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/microarray
workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/microarray/aux
workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/microarray/aux/qc
workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/microarray/hail.mt
workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/microarray/hail.mt/cols
workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/microarray/hail.mt/entries
workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/microarray/hail.mt/globals
workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/microarray/hail.mt/index
workspace/D

In [24]:
cmd = f"""
find {v9q} -maxdepth 6 -type f 2>/dev/null \
| grep -Ei '(\.pgen$|\.pvar$|\.pvar\.zst$|\.psam$|ancestry|pca|eigen|score)' \
| head -300
"""

res = subprocess.run(
    ["bash", "-lc", cmd],
    capture_output=True,
    text=True
)

print(res.stdout)

workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/acaf_threshold/pgen/acaf_threshold.chr1.pgen
workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/acaf_threshold/pgen/acaf_threshold.chr1.psam
workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/acaf_threshold/pgen/acaf_threshold.chr1.pvar
workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/acaf_threshold/pgen/acaf_threshold.chr10.pgen
workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/acaf_threshold/pgen/acaf_threshold.chr10.psam
workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/acaf_threshold/pgen/acaf_threshold.chr10.pvar
workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/wgs/short_read/s

In [25]:
from pathlib import Path
import subprocess
import pandas as pd

HOME = Path.home()
WS = HOME / "workspace"

GENOMICS = next(
    WS.glob("Data from All of Us Controlled Tier */vwb-aou-datasets-controlled-v9")
)

OUTROOT = WS / "agemr-data"

PGEN_DIR = (
    GENOMICS
    / "v9"
    / "wgs"
    / "short_read"
    / "snpindel"
    / "acaf_threshold"
    / "pgen"
)

EXTRACT_BED = HOME / "BMI_hg38_extract.bed"

print("GENOMICS =", GENOMICS)
print("PGEN_DIR exists =", PGEN_DIR.exists())
print("BED exists =", EXTRACT_BED.exists())

GENOMICS = /home/dataproc/workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9
PGEN_DIR exists = True
BED exists = True


In [26]:
chr1_prefix = PGEN_DIR / "acaf_threshold.chr1"

for ext in [".pgen", ".pvar", ".psam"]:
    f = Path(str(chr1_prefix) + ext)
    print(ext, f.exists(), f)

.pgen True /home/dataproc/workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/acaf_threshold/pgen/acaf_threshold.chr1.pgen
.pvar True /home/dataproc/workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/acaf_threshold/pgen/acaf_threshold.chr1.pvar
.psam True /home/dataproc/workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/acaf_threshold/pgen/acaf_threshold.chr1.psam


In [27]:
with open(str(chr1_prefix) + ".pvar") as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i >= 10:
            break

##FILTER=<ID=EXCESS_ALLELES,Description="Site has an excess of alternate alleles based on the input threshold">
##FILTER=<ID=ExcessHet,Description="Site has excess het value larger than the threshold">
##FILTER=<ID=LowQual,Description="Low quality">
##FILTER=<ID=NO_HQ_GENOTYPES,Description="Site has no high quality variant genotypes">
##INFO=<ID=AC,Number=A,Type=Integer,Description="Allele count in genotypes, for each ALT allele, in the same order as listed">
##INFO=<ID=AF,Number=A,Type=Float,Description="Allele Frequency, for each ALT allele, in the same order as listed">
##INFO=<ID=AN,Number=1,Type=Integer,Description="Total number of alleles in called genotypes">
##INFO=<ID=AS_QUALapprox,Number=1,Type=String,Description="Allele-specific QUAL approximations">
##INFO=<ID=CALIBRATION_SENSITIVITY,Number=A,Type=String,Description="Calibration sensitivity corresponding to the value of SCORE">
##INFO=<ID=QUALapprox,Number=1,Type=Integer,Description="Sum of PL[0] values; used to approximate

In [28]:
with open(str(chr1_prefix) + ".psam") as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i >= 5:
            break

#IID	SEX
1000000	NA
1000004	NA
1000033	NA
1000036	NA
1000039	NA


In [29]:
target_chr1 = bed_main.loc[bed_main["chrom"] == "chr1"]

print("Target chr1 SNPs:", len(target_chr1))
display(target_chr1.head())

Target chr1 SNPs: 261


,chrom,start0,end0,rsid
7,chr1,177919889,177919890,rs539515
20,chr1,201831382,201831383,rs2678204
27,chr1,72284786,72284787,rs34361149
44,chr1,109612065,109612066,rs17024393
70,chr1,49317099,49317100,rs241460


In [32]:
import shutil
from pathlib import Path

for cmd in ["plink2", "plink"]:
    print(cmd, "->", shutil.which(cmd))

for p in [
    "/bin/plink2",
    "/usr/bin/plink2",
    "/usr/local/bin/plink2",
    "/bin/plink",
    "/usr/bin/plink",
    "/usr/local/bin/plink",
]:
    print(p, Path(p).exists())

plink2 -> None
plink -> None
/bin/plink2 False
/usr/bin/plink2 False
/usr/local/bin/plink2 False
/bin/plink False
/usr/bin/plink False
/usr/local/bin/plink False


In [33]:
import shutil
import subprocess
from pathlib import Path

print("conda:", shutil.which("conda"))
print("mamba:", shutil.which("mamba"))

conda: /opt/conda/default/bin/conda
mamba: /opt/conda/default/bin/mamba


In [35]:
from pathlib import Path
import subprocess

PLINK_ENV = Path.home() / "plink2_env"
MAMBA = "/opt/conda/default/bin/mamba"

cmd = [
    MAMBA,
    "create",
    "-y",
    "-p", str(PLINK_ENV),
    "-c", "conda-forge",
    "-c", "bioconda",
    "plink2"
]

print(" ".join(cmd))

res = subprocess.run(
    cmd,
    text=True,
    capture_output=True
)

print(res.stdout)
print(res.stderr)
print("return code =", res.returncode)

/opt/conda/default/bin/mamba create -y -p /home/dataproc/plink2_env -c conda-forge -c bioconda plink2
Transaction

  Prefix: /home/dataproc/plink2_env

  Updating specs:

   - plink2


  Package             Version  Build       Channel          Size
──────────────────────────────────────────────────────────────────
  Install:
──────────────────────────────────────────────────────────────────

  + libgomp            16.2.0  he0feb66_7  conda-forge     641kB
  + _openmp_mutex         4.5  20_gnu      conda-forge      29kB
  + libgcc             16.2.0  ha9f2e26_7  conda-forge       1MB
  + libstdcxx          16.2.0  h934c35e_7  conda-forge       7MB
  + plink2         2.0.0a.6.9  h9948957_0  bioconda          8MB

  Summary:

  Install: 5 packages

  Total download: 16MB

──────────────────────────────────────────────────────────────────



Looking for: ['plink2']


Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... do

In [36]:
PLINK2 = PLINK_ENV / "bin" / "plink2"

print("PLINK2 exists:", PLINK2.exists())

res = subprocess.run(
    [str(PLINK2), "--version"],
    text=True,
    capture_output=True
)

print(res.stdout)
print(res.stderr)

PLINK2 exists: True
PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)




In [37]:
pvar_file = Path(str(chr1_prefix) + ".pvar")

shown = 0

with open(pvar_file) as f:
    for line in f:
        if line.startswith("##"):
            continue

        print(line.rstrip())
        shown += 1

        if shown >= 5:
            break

#CHROM	POS	ID	REF	ALT	FILTER	INFO
chr1	10060	chr1:10060:CTA	CTA	CA,CTTA,ATA,C	.	AC=310,25,2,3;AF=3.320e-04,2.677e-05,2.142e-06,3.213e-06;AN=933832;AS_QUALapprox=0|7313|461|139|102;CALIBRATION_SENSITIVITY=0.9897,0.9739,0.9963,0.9825;QUALapprox=25;SCORE=-0.6671,-0.6046,-0.6872,-0.6329
chr1	10061	chr1:10061:TA	TA	TAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCA,AA,TAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCTAACCCAACCCTAACCCTAACCCTAACCCTAACCCTAACCCA,T,CA,TGA,TAA,GA	.	AC=20,7,1,70,35,1,23,4;AF=4.114e-03,1.440e-03,2.057e-04,0.014,7.199e-03,2.057e-04,4.731e-03,8.227e-04;AN=4862;AS_QUALapprox=0|468|346|10|1186|1642|2|378|124;CALIBRATION_SENSITIVITY=0.9846,0.9942,0.9871,0.9556,0.9963,0.9879,0.9816,0.9967;QUALapprox=25;SCORE=-0.6419,-0.6585,-0.6531,-0.5622,-0.6875,-0.6575,-0.6295,-0.6961
chr1	10066	chr1:10066:CTA	CTA	CA,CGTA,CTTA,ATA,C	.	AC=240,1,17,1,1;AF=2.479e-04,1.033e-06,1.756e-05,1.033e-06,1.033e-06;AN=967972;AS_QUALapprox=0|5578|4|298|77|22;CALIBRATION_SENSITIVITY=0.9881,0.9825,0.974,0.99

In [38]:
TEST_DIR = Path.home() / "bmi_extract_test"
TEST_DIR.mkdir(exist_ok=True)

test_out = TEST_DIR / "BMI_chr1"

cmd = [
    str(PLINK2),
    "--pfile", str(chr1_prefix),
    "--extract", "bed0", str(EXTRACT_BED),
    "--snps-only", "just-acgt",
    "--max-alleles", "2",
    "--make-pgen",
    "--out", str(test_out)
]

print(" ".join(cmd))

res = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(res.stdout)
print(res.stderr)
print("return code =", res.returncode)

/home/dataproc/plink2_env/bin/plink2 --pfile /home/dataproc/workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/acaf_threshold/pgen/acaf_threshold.chr1 --extract bed0 /home/dataproc/BMI_hg38_extract.bed --snps-only just-acgt --max-alleles 2 --make-pgen --out /home/dataproc/bmi_extract_test/BMI_chr1
PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/dataproc/bmi_extract_test/BMI_chr1.log.
Options in effect:
  --extract bed0 /home/dataproc/BMI_hg38_extract.bed
  --make-pgen
  --max-alleles 2
  --out /home/dataproc/bmi_extract_test/BMI_chr1
  --pfile /home/dataproc/workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/acaf_threshold/pgen/acaf_threshold.chr1
  --snps-only just-acgt

Start time: Wed Sep 30 16:23:40 2026
7950 MiB RAM detected, ~2064 available; reservi

In [39]:
out_pvar = Path(str(test_out) + ".pvar")

rows = []

with open(out_pvar) as f:
    for line in f:
        if line.startswith("##"):
            continue
        if line.startswith("#CHROM"):
            header = line.rstrip().split("\t")
            continue
        rows.append(line.rstrip().split("\t"))

out_chr1 = pd.DataFrame(rows, columns=header)

print("Target chr1 loci:", len(target_chr1))
print("Extracted chr1 variants:", len(out_chr1))

display(out_chr1.head())

Target chr1 loci: 261
Extracted chr1 variants: 172


,#CHROM,POS,ID,REF,ALT,INFO
0,1,1756945,chr1:1756945:G,G,A,AC=60287;AF=0.056;AN=1071266;AS_QUALapprox=0|5...
1,1,3189741,chr1:3189741:A,A,G,AC=681374;AF=0.636;AN=1071242;AS_QUALapprox=0|...
2,1,4172101,chr1:4172101:C,C,T,AC=299786;AF=0.280;AN=1071228;AS_QUALapprox=0|...
3,1,4497257,chr1:4497257:C,C,T,AC=273206;AF=0.255;AN=1071244;AS_QUALapprox=0|...
4,1,5126191,chr1:5126191:C,C,T,AC=6444;AF=6.015e-03;AN=1071270;AS_QUALapprox=...


In [40]:
out_chr1["POS"] = out_chr1["POS"].astype(int)

dup_pos = (
    out_chr1.groupby(["#CHROM", "POS"])
    .size()
    .reset_index(name="n")
    .query("n > 1")
)

print("Target positions with >1 extracted biallelic variant:",
      len(dup_pos))

display(dup_pos.head(20))

Target positions with >1 extracted biallelic variant: 0


,#CHROM,POS,n


In [41]:
target_pos = set(target_chr1["end0"].astype(int))
found_pos = set(out_chr1["POS"].astype(int))

missing_pos = sorted(target_pos - found_pos)

print("Target:", len(target_pos))
print("Found positions:", len(target_pos & found_pos))
print("Missing positions:", len(missing_pos))
print("First missing:", missing_pos[:20])

Target: 261
Found positions: 172
Missing positions: 89
First missing: [1659060, 2118589, 2496273, 3696591, 8630874, 13546621, 13582625, 18808340, 21608865, 24592632, 27504554, 30145836, 31728046, 33319328, 33837072, 34439602, 35222940, 39099258, 47204853, 53283216]


In [42]:
raw_out = TEST_DIR / "BMI_chr1_raw"

cmd = [
    str(PLINK2),
    "--pfile", str(chr1_prefix),
    "--extract", "bed0", str(EXTRACT_BED),
    "--make-pgen",
    "--out", str(raw_out)
]

res = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(res.stdout)
print(res.stderr)
print("return code =", res.returncode)

PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /home/dataproc/bmi_extract_test/BMI_chr1_raw.log.
Options in effect:
  --extract bed0 /home/dataproc/BMI_hg38_extract.bed
  --make-pgen
  --out /home/dataproc/bmi_extract_test/BMI_chr1_raw
  --pfile /home/dataproc/workspace/Data from All of Us Controlled Tier /vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/acaf_threshold/pgen/acaf_threshold.chr1

Start time: Wed Sep 30 16:26:28 2026
7950 MiB RAM detected, ~2058 available; reserving 1994 MiB for main workspace.
Using up to 2 compute threads.
535662 samples (0 females, 0 males, 535662 ambiguous; 535662 founders) loaded
from /home/dataproc/workspace/Data from All of Us Controlled Tier
/vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/acaf_threshold/pgen/acaf_threshold.chr1.psam.
4792392 variants loaded from /home/dataproc/workspace/Data from All of Us
C

In [43]:
raw_chr1["POS"] = raw_chr1["POS"].astype(int)

raw_found_pos = set(raw_chr1["POS"])
target_pos = set(target_chr1["end0"].astype(int))

print("Raw target positions found:",
      len(target_pos & raw_found_pos))

print("Still absent from ACAF:",
      len(target_pos - raw_found_pos))

NameError: name 'raw_chr1' is not defined

In [44]:
raw_pvar = Path(str(raw_out) + ".pvar")

rows = []

with open(raw_pvar) as f:
    for line in f:
        if line.startswith("##"):
            continue

        if line.startswith("#CHROM"):
            header = line.rstrip().split("\t")
            continue

        rows.append(line.rstrip().split("\t"))

raw_chr1 = pd.DataFrame(rows, columns=header)

raw_chr1["POS"] = raw_chr1["POS"].astype(int)

print("Raw extracted records:", len(raw_chr1))

display(
    raw_chr1[
        ["#CHROM", "POS", "ID", "REF", "ALT"]
    ].head(20)
)

Raw extracted records: 260


,#CHROM,POS,ID,REF,ALT
0,1,1659060,chr1:1659060:GACA,GACA,"AACA,G,CACA"
1,1,1756945,chr1:1756945:G,G,A
2,1,2118589,chr1:2118589:G,G,"A,T"
3,1,2496273,chr1:2496273:C,C,"T,A"
4,1,3189741,chr1:3189741:A,A,G
5,1,3696591,chr1:3696591:C,C,"G,T"
6,1,4172101,chr1:4172101:C,C,T
7,1,4497257,chr1:4497257:C,C,T
8,1,5126191,chr1:5126191:C,C,T
9,1,6637807,chr1:6637807:A,A,G


In [45]:
target_pos = set(target_chr1["end0"].astype(int))
raw_found_pos = set(raw_chr1["POS"])

print("Target loci:", len(target_pos))
print("Found target positions:", len(target_pos & raw_found_pos))
print("Still absent from ACAF:", len(target_pos - raw_found_pos))

missing_pos = sorted(target_pos - raw_found_pos)
print("Missing positions:", missing_pos)

Target loci: 261
Found target positions: 260
Still absent from ACAF: 1
Missing positions: [109469373]


In [46]:
dup_pos_raw = (
    raw_chr1
    .groupby(["#CHROM", "POS"])
    .size()
    .reset_index(name="n")
    .query("n > 1")
)

print("Positions with >1 AoU record:", len(dup_pos_raw))
display(dup_pos_raw)

Positions with >1 AoU record: 0


,#CHROM,POS,n


In [47]:
from pathlib import Path
import pandas as pd
import subprocess
import shutil

LOCAL_DIR = Path.home() / "BMI_AoU_ACAF_extract"
LOCAL_DIR.mkdir(exist_ok=True)

SAVE_DIR = OUTROOT / "genotypes" / "BMI_AoU_ACAF_v9"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

summaries = []
mappings = []
missings = []

for chrom_num in range(1, 23):

    chrom = f"chr{chrom_num}"

    print(f"\n========== {chrom} ==========")

    # Target SNPs on this chromosome
    target = bed_main.loc[
        bed_main["chrom"] == chrom
    ].copy()

    target["POS"] = target["end0"].astype(int)

    # chromosome-specific BED
    chr_bed = LOCAL_DIR / f"target_{chrom}.bed"

    target[
        ["chrom", "start0", "end0", "rsid"]
    ].to_csv(
        chr_bed,
        sep="\t",
        header=False,
        index=False
    )

    # AoU ACAF PGEN prefix
    input_prefix = (
        PGEN_DIR / f"acaf_threshold.{chrom}"
    )

    out_prefix = LOCAL_DIR / f"BMI_{chrom}"

    # Extract all records at target positions
    cmd = [
        str(PLINK2),
        "--pfile", str(input_prefix),
        "--extract", "bed0", str(chr_bed),
        "--make-pgen",
        "--out", str(out_prefix)
    ]

    res = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    if res.returncode != 0:
        print(res.stdout)
        print(res.stderr)
        raise RuntimeError(f"PLINK failed on {chrom}")

    # Read output PVAR
    pvar_file = Path(str(out_prefix) + ".pvar")

    rows = []

    with open(pvar_file) as f:
        for line in f:

            if line.startswith("##"):
                continue

            if line.startswith("#CHROM"):
                header = line.rstrip().split("\t")
                continue

            rows.append(line.rstrip().split("\t"))

    aou = pd.DataFrame(rows, columns=header)

    if len(aou) > 0:
        aou["POS"] = aou["POS"].astype(int)

    target_pos = set(target["POS"])
    found_pos = set(aou["POS"]) if len(aou) else set()

    missing_pos = target_pos - found_pos

    print(
        f"Target={len(target_pos)}, "
        f"Found={len(target_pos & found_pos)}, "
        f"Missing={len(missing_pos)}, "
        f"AoU records={len(aou)}"
    )

    # Summary
    summaries.append({
        "chrom": chrom,
        "target": len(target_pos),
        "found": len(target_pos & found_pos),
        "missing": len(missing_pos),
        "aou_records": len(aou)
    })

    # Create rsID -> AoU record mapping
    if len(aou):

        keep_cols = [
            "#CHROM", "POS", "ID",
            "REF", "ALT"
        ]

        tmp = target.merge(
            aou[keep_cols],
            on="POS",
            how="left"
        )

    else:
        tmp = target.copy()

        for c in [
            "#CHROM", "ID",
            "REF", "ALT"
        ]:
            tmp[c] = pd.NA

    mappings.append(tmp)

    # missing SNPs
    if missing_pos:

        miss = target.loc[
            target["POS"].isin(missing_pos)
        ].copy()

        missings.append(miss)

    # Persist PLINK files immediately
    for ext in [
        ".pgen",
        ".pvar",
        ".psam",
        ".log"
    ]:

        src = Path(str(out_prefix) + ext)

        if src.exists():
            shutil.copy2(
                src,
                SAVE_DIR / src.name
            )


========== chr1 ==========
Target=261, Found=260, Missing=1, AoU records=260

========== chr2 ==========
Target=254, Found=253, Missing=1, AoU records=253

========== chr3 ==========
Target=229, Found=229, Missing=0, AoU records=229

========== chr4 ==========
Target=228, Found=228, Missing=0, AoU records=228

========== chr5 ==========
Target=203, Found=203, Missing=0, AoU records=203

========== chr6 ==========
Target=209, Found=209, Missing=0, AoU records=209

========== chr7 ==========
Target=157, Found=157, Missing=0, AoU records=157

========== chr8 ==========
Target=205, Found=205, Missing=0, AoU records=205

========== chr9 ==========
Target=119, Found=119, Missing=0, AoU records=119

========== chr10 ==========
Target=141, Found=141, Missing=0, AoU records=141

========== chr11 ==========
Target=130, Found=130, Missing=0, AoU records=130

========== chr12 ==========
Target=142, Found=142, Missing=0, AoU records=142

========== chr13 ==========
Target=90, Found=90, Missing=0, 

In [48]:
summary = pd.DataFrame(summaries)

display(summary)

print(
    "\nTotal target:",
    summary["target"].sum()
)

print(
    "Total found:",
    summary["found"].sum()
)

print(
    "Total missing:",
    summary["missing"].sum()
)

,chrom,target,found,missing,aou_records
0,chr1,261,260,1,260
1,chr2,254,253,1,253
2,chr3,229,229,0,229
3,chr4,228,228,0,228
4,chr5,203,203,0,203
5,chr6,209,209,0,209
6,chr7,157,157,0,157
7,chr8,205,205,0,205
8,chr9,119,119,0,119
9,chr10,141,141,0,141



Total target: 3023
Total found: 3018
Total missing: 5


In [49]:
mapping = pd.concat(
    mappings,
    ignore_index=True
)

mapping_out = SAVE_DIR / "BMI_target_to_AoU_variant_manifest.tsv"

mapping.to_csv(
    mapping_out,
    sep="\t",
    index=False
)

display(
    mapping[
        [
            "rsid",
            "chrom",
            "POS",
            "ID",
            "REF",
            "ALT"
        ]
    ].head(20)
)

,rsid,chrom,POS,ID,REF,ALT
0,rs539515,chr1,177919890,chr1:177919890:A,A,"C,T"
1,rs2678204,chr1,201831383,chr1:201831383:T,T,G
2,rs34361149,chr1,72284787,chr1:72284787:G,G,"A,T"
3,rs17024393,chr1,109612066,chr1:109612066:T,T,C
4,rs241460,chr1,49317100,chr1:49317100:A,A,G
5,rs61813324,chr1,156080086,chr1:156080086:C,C,T
6,rs79113395,chr1,1659060,chr1:1659060:GACA,GACA,"AACA,G,CACA"
7,rs12140153,chr1,62114219,chr1:62114219:G,G,T
8,rs815163,chr1,190325596,chr1:190325596:T,T,C
9,rs588660,chr1,96421048,chr1:96421048:G,G,"A,T"


In [50]:
if missings:

    missing_all = pd.concat(
        missings,
        ignore_index=True
    )

else:

    missing_all = pd.DataFrame(
        columns=bed_main.columns
    )

missing_all.to_csv(
    SAVE_DIR / "BMI_target_missing_from_ACAF.tsv",
    sep="\t",
    index=False
)

summary.to_csv(
    SAVE_DIR / "BMI_ACAF_extraction_summary.tsv",
    sep="\t",
    index=False
)

print("Missing SNPs:", len(missing_all))
print("Saved to:", SAVE_DIR)

Missing SNPs: 5
Saved to: /home/dataproc/workspace/agemr-data/genotypes/BMI_AoU_ACAF_v9


In [51]:
ANCESTRY_FILE = (
    GENOMICS
    / "v9"
    / "wgs"
    / "short_read"
    / "snpindel"
    / "aux"
    / "ancestry"
    / "ancestry_preds.tsv"
)

print(ANCESTRY_FILE.exists())

True


In [52]:
import pandas as pd
import numpy as np
import ast

ANCESTRY_FILE = (
    GENOMICS
    / "v9"
    / "wgs"
    / "short_read"
    / "snpindel"
    / "aux"
    / "ancestry"
    / "ancestry_preds.tsv"
)

anc = pd.read_csv(
    ANCESTRY_FILE,
    sep="\t",
    dtype={"research_id": str}
)

print(anc.shape)
print(anc.columns)
display(anc.head())

print(
    anc["ancestry_pred_other"]
    .value_counts(dropna=False)
)

(535662, 5)
Index(['research_id', 'ancestry_pred', 'probabilities', 'pca_features',
       'ancestry_pred_other'],
      dtype='object')


,research_id,ancestry_pred,probabilities,pca_features,ancestry_pred_other
0,1000000,afr,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[-0.28319043946421235, -0.005560140576153653, ...",afr
1,1000004,eur,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0]","[0.10662426188064174, 0.1406680230277923, 0.00...",eur
2,1000033,eur,"[0.01, 0.0, 0.0, 0.99, 0.0, 0.0]","[0.10378596780407301, 0.12484972382368799, 0.0...",eur
3,1000036,afr,"[0.91, 0.04, 0.0, 0.02, 0.03, 0.0]","[-0.17521518623309146, 0.03332447908941083, 0....",afr
4,1000039,afr,"[0.97, 0.02, 0.01, 0.0, 0.0, 0.0]","[-0.2574299552888707, 0.005215228761592666, 0....",afr


ancestry_pred_other
eur    284004
amr     97261
afr     94465
oth     40029
eas     12980
sas      5631
mid      1292
Name: count, dtype: int64


In [53]:
def parse_pc(x):
    if isinstance(x, str):
        return np.asarray(ast.literal_eval(x), dtype=float)
    return np.asarray(x, dtype=float)

pc_array = np.vstack(
    anc["pca_features"].map(parse_pc)
)

print(pc_array.shape)

(535662, 16)


In [54]:
pc_df = pd.DataFrame(
    pc_array,
    columns=[f"PC{i}" for i in range(1, 17)]
)

covar = pd.concat(
    [
        anc[
            [
                "research_id",
                "ancestry_pred",
                "ancestry_pred_other"
            ]
        ].reset_index(drop=True),
        pc_df
    ],
    axis=1
)

display(covar.head())

,research_id,ancestry_pred,ancestry_pred_other,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16
0,1000000,afr,afr,-0.283190,-0.005560,0.000661,0.004305,0.008885,0.025120,-0.005581,0.000184,-0.005552,-0.003064,0.000549,0.002371,0.004154,-0.004973,-0.006457,0.005226
1,1000004,eur,eur,0.106624,0.140668,0.007041,0.053305,-0.015939,0.002309,0.006480,-0.019581,-0.012097,0.000399,-0.000863,0.001100,-0.003762,0.001690,0.000398,0.002015
2,1000033,eur,eur,0.103786,0.124850,0.009130,0.040755,-0.011007,0.003702,0.003614,-0.026160,-0.018172,0.001075,0.000326,0.000588,-0.001113,-0.001235,-0.000339,-0.000004
3,1000036,afr,afr,-0.175215,0.033324,0.005418,0.014608,0.000177,0.016215,-0.001289,-0.000835,-0.014240,-0.001628,0.003646,0.005642,0.001095,-0.005902,0.005258,-0.001938
4,1000039,afr,afr,-0.257430,0.005215,0.001636,0.001427,-0.003476,0.031687,0.007957,-0.003891,-0.002539,-0.016037,-0.001653,0.005505,-0.005676,0.001279,-0.001427,0.003998


In [55]:
COV_DIR = OUTROOT / "covariates"
COV_DIR.mkdir(parents=True, exist_ok=True)

covar.to_csv(
    COV_DIR / "AoU_v9_ancestry_PC1_PC16.tsv",
    sep="\t",
    index=False
)

In [56]:
print(COV_DIR)

/home/dataproc/workspace/agemr-data/covariates


In [57]:
eur = covar.loc[
    covar["ancestry_pred_other"].str.lower() == "eur"
].copy()

print("Predicted EUR:", len(eur))

Predicted EUR: 284004


In [58]:
eur.to_csv(
    COV_DIR / "AoU_v9_EUR_PC1_PC16.tsv",
    sep="\t",
    index=False
)

In [59]:
SNAP = next(
    (Path.home() / "workspace").glob(
        "Data from All of Us Controlled Tier /wgs_bmi_bp"
    )
)

print(SNAP)
print(SNAP.exists())

for x in SNAP.iterdir():
    print(x.name)

StopIteration: 